# LeetCode 367 — Valid Perfect Square bằng Python

**Link đề:** https://leetcode.com/problems/valid-perfect-square/
**Độ khó:** Easy · **Chủ đề:** Math, Binary Search

```python
class Solution:
    def isPerfectSquare(self, num: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **số chính phương (perfect square)** và vì sao đề **cấm** dùng `sqrt`.
2. Đi từ cách thử lần lượt `O(√n)` → **tìm nhị phân trên đáp án (binary search on answer)** `O(log n)`.
3. Biết hai cách sáng tạo: **tổng các số lẻ liên tiếp** và **phương pháp Newton**.
4. Trace từng bước `lo`, `hi`, `mid`; chứng minh vòng lặp dừng và trả đúng.
5. Tránh lỗi kinh điển: vòng lặp vô hạn, sai biên, sai số dấu chấm động (float).

## 1. Đề bài

Cho một số nguyên **dương** `num`. Trả `True` nếu `num` là **số chính phương**, ngược lại `False`.

Số chính phương là số bằng **bình phương của một số nguyên**: `num = k * k` với `k` nguyên.

**Không được** dùng hàm thư viện có sẵn như `sqrt`.

| `num` | Kết quả | Vì sao |
|---:|:---:|---|
| `16` | `True` | `4 * 4 = 16` |
| `14` | `False` | `√14 ≈ 3.742` không nguyên; `3*3 = 9 < 14 < 16 = 4*4` |
| `1` | `True` | `1 * 1 = 1` |
| `2147395600` | `True` | `46340 * 46340` |

**Constraints:** `1 <= num <= 2^31 - 1`.

In [1]:
# Liệt kê vài số chính phương đầu tiên để có cảm giác
squares = [k * k for k in range(1, 11)]
print(squares)
print("16 có trong danh sách?", 16 in squares, "| 14?", 14 in squares)
print("2^31 - 1 =", 2**31 - 1)

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
16 có trong danh sách? True | 14? False
2^31 - 1 = 2147483647


**Quan sát then chốt:** dãy `k*k` **tăng dần** theo `k`. Vì vậy `num` là chính phương ⇔ tồn tại `k` với `k*k == num`; nếu đi qua `num` (`k*k > num`) thì không cần thử tiếp. Tính "tăng dần" này mở đường cho tìm nhị phân.

## 2. Nền tảng Python

### 2.1. Phép nhân, lũy thừa và chia nguyên

- `k * k` hoặc `k ** 2` — bình phương.
- `a // b` — chia lấy phần **nguyên** (làm tròn xuống). `a / b` — chia ra **số thực (float)**.
- `(lo + hi) // 2` — điểm giữa nguyên.

In [2]:
print(7 * 7, 7 ** 2)
print("15 / 2 =", 15 / 2, "| 15 // 2 =", 15 // 2)
lo, hi = 1, 16
print("mid =", (lo + hi) // 2)

49 49
15 / 2 = 7.5 | 15 // 2 = 7
mid = 8


### 2.2. Vì sao không tin `float` cho số lớn?

`num ** 0.5` tính căn bậc hai bằng **số thực dấu chấm động**, chỉ có khoảng 15–17 chữ số chính xác. Với số rất lớn, kết quả có thể lệch một chút, `int(...)` có thể làm tròn sai. Đề cũng **cấm** cách này; ta học nó chỉ để thấy rủi ro.

In [3]:
x = 16 ** 0.5
print(x, type(x), x == int(x))

big = (10**8 + 1) ** 2 - 1            # không phải chính phương
r = big ** 0.5
print("big ** 0.5 =", r, "| int =", int(r), "| int(r)**2 == big?", int(r) ** 2 == big)
print("float cho rằng căn là số nguyên?", r == int(r), "<- SAI, big không phải chính phương!")

4.0 <class 'float'> True
big ** 0.5 = 100000001.0 | int = 100000001 | int(r)**2 == big? False
float cho rằng căn là số nguyên? True <- SAI, big không phải chính phương!


Ở trên, `big ** 0.5` được làm tròn thành đúng `100000001.0`, khiến kiểm tra "căn có nguyên không" kết luận sai. Với constraints `num ≤ 2^31 - 1` thì float thường vẫn đúng, nhưng thói quen tốt là **chỉ dùng số nguyên**.

### 2.3. Vòng `while`

`while điều_kiện:` lặp **khi** điều kiện còn đúng. Phải chắc chắn điều kiện **sẽ sai** tại một lúc nào đó, nếu không sẽ lặp vô hạn.

In [4]:
k = 1
while k * k <= 30:
    print(f"k={k}, k*k={k*k}")
    k += 1
print("Dừng vì", k, "*", k, "=", k * k, "> 30")

k=1, k*k=1
k=2, k*k=4
k=3, k*k=9
k=4, k*k=16
k=5, k*k=25
Dừng vì 6 * 6 = 36 > 30


## 3. Cách 1 — Trực giác: thử lần lượt `k = 1, 2, 3, ...`

Thử từng `k` cho tới khi `k*k >= num`:
- `k*k == num` → `True`.
- `k*k > num` → đã vượt, không còn hy vọng → `False`.

In [5]:
def is_perfect_square_linear(num: int) -> bool:
    k = 1
    while k * k < num:
        k += 1
    return k * k == num

for n in [1, 4, 14, 16, 25, 26]:
    print(n, "->", is_perfect_square_linear(n))

1 -> True
4 -> True
14 -> False
16 -> True
25 -> True
26 -> False


### 3.1. Nút thắt

Số vòng lặp ≈ `√num`. Với `num ≈ 2.1·10^9`, đó là ~**46 341** vòng — vẫn chạy được, nhưng nếu giới hạn là `10^18` thì cần `10^9` vòng → quá chậm.

**Insight:** dãy `1², 2², 3², ...` **tăng dần**. Thay vì đi từng bước, ta có thể **đoán ở giữa** khoảng `[1, num]`, so `mid*mid` với `num`, rồi **bỏ một nửa** khoảng — đúng như trò chơi đoán số. Đó là **binary search**, cần khoảng `log₂(num) ≈ 31` bước.

In [6]:
import math
for n in [10**4, 2**31 - 1, 10**18]:
    print(f"num={n}: tuyến tính ~{math.isqrt(n):,} vòng | nhị phân ~{n.bit_length()} vòng")

num=10000: tuyến tính ~100 vòng | nhị phân ~14 vòng
num=2147483647: tuyến tính ~46,340 vòng | nhị phân ~31 vòng
num=1000000000000000000: tuyến tính ~1,000,000,000 vòng | nhị phân ~60 vòng


(Ở cell trên dùng `math.isqrt` và `int.bit_length()` **chỉ để đếm minh họa**, không dùng trong lời giải. `n.bit_length()` = số bit cần để biểu diễn `n` ≈ `log₂ n`.)

## 4. Cách 2 — Binary search trên đáp án (lời giải chính)

Tìm `k` trong khoảng `[lo, hi] = [1, num]` sao cho `k*k == num`.

**Đoạn 1 — khởi tạo:** `lo = 1`, `hi = num`. Đáp án (nếu có) chắc chắn nằm trong `[lo, hi]`.

**Đoạn 2 — lấy giữa:** `mid = (lo + hi) // 2`, tính `sq = mid * mid`.

**Đoạn 3 — so sánh và bỏ nửa:**
- `sq == num` → tìm thấy → `True`.
- `sq < num` → `mid` quá nhỏ; mọi `k ≤ mid` cũng quá nhỏ → `lo = mid + 1`.
- `sq > num` → `mid` quá lớn → `hi = mid - 1`.

**Đoạn 4 — dừng:** khi `lo > hi` khoảng rỗng → không có `k` → `False`.

In [7]:
# Đoạn 2-3 thử bằng tay một bước với num = 16
lo, hi, num = 1, 16, 16
mid = (lo + hi) // 2
print(f"mid={mid}, mid*mid={mid*mid} -> {'quá lớn, hi = mid - 1' if mid*mid > num else 'quá nhỏ hoặc bằng'}")

mid=8, mid*mid=64 -> quá lớn, hi = mid - 1


In [8]:
def is_perfect_square_binary(num: int) -> bool:
    lo, hi = 1, num
    while lo <= hi:
        mid = (lo + hi) // 2
        sq = mid * mid
        if sq == num:
            return True
        if sq < num:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

for n in [1, 4, 14, 16, 25, 26, 2147395600, 2**31 - 1]:
    print(n, "->", is_perfect_square_binary(n))

1 -> True
4 -> True
14 -> False
16 -> True
25 -> True
26 -> False
2147395600 -> True
2147483647 -> False


## 5. TRACE từng bước

### 5.1. `num = 16` (có đáp án)

| Bước | `lo` | `hi` | `mid` | `mid*mid` | So sánh | Hành động |
|---:|---:|---:|---:|---:|---|---|
| 1 | 1 | 16 | 8 | 64 | > 16 | `hi = 7` |
| 2 | 1 | 7 | 4 | 16 | == 16 | **True** |

### 5.2. `num = 14` (không có đáp án)

| Bước | `lo` | `hi` | `mid` | `mid*mid` | So sánh | Hành động |
|---:|---:|---:|---:|---:|---|---|
| 1 | 1 | 14 | 7 | 49 | > 14 | `hi = 6` |
| 2 | 1 | 6 | 3 | 9 | < 14 | `lo = 4` |
| 3 | 4 | 6 | 5 | 25 | > 14 | `hi = 4` |
| 4 | 4 | 4 | 4 | 16 | > 14 | `hi = 3` |
| — | 4 | 3 | | | `lo > hi` | **False** |

Nhánh đáng chú ý ở 5.2: khoảng co dần đến **rỗng**; lúc dừng `hi = 3 = ⌊√14⌋` và `lo = 4 = hi + 1`.

In [9]:
def trace_binary(num: int) -> bool:
    lo, hi = 1, num
    step = 1
    print(f"num = {num}")
    while lo <= hi:
        mid = (lo + hi) // 2
        sq = mid * mid
        if sq == num:
            print(f"Bước {step}: lo={lo}, hi={hi}, mid={mid}, mid²={sq} == num -> True")
            return True
        if sq < num:
            print(f"Bước {step}: lo={lo}, hi={hi}, mid={mid}, mid²={sq} < num -> lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"Bước {step}: lo={lo}, hi={hi}, mid={mid}, mid²={sq} > num -> hi = {mid - 1}")
            hi = mid - 1
        step += 1
    print(f"Dừng: lo={lo} > hi={hi} -> False")
    return False

_ = trace_binary(16)
print()
_ = trace_binary(14)

num = 16
Bước 1: lo=1, hi=16, mid=8, mid²=64 > num -> hi = 7
Bước 2: lo=1, hi=7, mid=4, mid²=16 == num -> True

num = 14
Bước 1: lo=1, hi=14, mid=7, mid²=49 > num -> hi = 6
Bước 2: lo=1, hi=6, mid=3, mid²=9 < num -> lo = 4
Bước 3: lo=4, hi=6, mid=5, mid²=25 > num -> hi = 4
Bước 4: lo=4, hi=4, mid=4, mid²=16 > num -> hi = 3
Dừng: lo=4 > hi=3 -> False


### 5.3. Thu hẹp khoảng ban đầu (tối ưu nhỏ)

Với `num ≥ 4`, `√num ≤ num // 2`, nên có thể đặt `hi = num // 2` (giảm 1 bước). Với `num = 1, 2, 3` phải xử lý riêng (vì `num // 2` có thể bằng `0` hoặc `1`). Tối ưu này không đổi Big-O; chỉ cần biết là có.

In [10]:
for n in [4, 9, 16, 100, 2147395600]:
    k = math.isqrt(n)     # chỉ để minh họa bất đẳng thức
    print(f"num={n}: sqrt={k}, num//2={n // 2}, sqrt <= num//2? {k <= n // 2}")

num=4: sqrt=2, num//2=2, sqrt <= num//2? True
num=9: sqrt=3, num//2=4, sqrt <= num//2? True
num=16: sqrt=4, num//2=8, sqrt <= num//2? True
num=100: sqrt=10, num//2=50, sqrt <= num//2? True
num=2147395600: sqrt=46340, num//2=1073697800, sqrt <= num//2? True


## 6. Cách 3 — Tổng các số lẻ liên tiếp

Sự thật toán học đẹp: **`1 + 3 + 5 + ... + (2k − 1) = k²`**.

| k | Tổng `k` số lẻ đầu | = k² |
|---:|---|---:|
| 1 | 1 | 1 |
| 2 | 1 + 3 | 4 |
| 3 | 1 + 3 + 5 | 9 |
| 4 | 1 + 3 + 5 + 7 | 16 |

Hình dung: một hình vuông `k×k` thêm một "góc chữ L" gồm `2k + 1` ô thì thành hình vuông `(k+1)×(k+1)`.

Thuật toán: trừ dần `1, 3, 5, ...` khỏi `num`; nếu về **đúng 0** thì chính phương, nếu xuống **âm** thì không.

In [11]:
def is_perfect_square_odd(num: int) -> bool:
    odd = 1
    while num > 0:
        num -= odd
        odd += 2
    return num == 0

def trace_odd(num):
    odd, step = 1, 1
    while num > 0:
        print(f"Bước {step}: num={num} - {odd} = {num - odd}")
        num -= odd
        odd += 2
        step += 1
    print("Kết thúc num =", num, "->", num == 0)

trace_odd(16)
trace_odd(14)

Bước 1: num=16 - 1 = 15
Bước 2: num=15 - 3 = 12
Bước 3: num=12 - 5 = 7
Bước 4: num=7 - 7 = 0
Kết thúc num = 0 -> True
Bước 1: num=14 - 1 = 13
Bước 2: num=13 - 3 = 10
Bước 3: num=10 - 5 = 5
Bước 4: num=5 - 7 = -2
Kết thúc num = -2 -> False


Cách này chỉ dùng phép trừ, rất "tay", nhưng vẫn `O(√num)` vòng như Cách 1.

## 7. Cách 4 — Phương pháp Newton (Newton's method)

Muốn tìm `x` sao cho `x² = num`. Newton cho công thức cải thiện dần một dự đoán `x`:

```text
x_mới = (x + num / x) / 2
```

Phiên bản **số nguyên**: `x = (x + num // x) // 2`. Bắt đầu từ `x = num` (luôn ≥ căn), mỗi bước `x` giảm dần và hội tụ **rất nhanh** về `⌊√num⌋`. Dừng khi `x * x <= num`.

Ví dụ `num = 16`: `16 → (16 + 1)//2 = 8 → (8 + 2)//2 = 5 → (5 + 3)//2 = 4` → `4*4 = 16 ≤ 16` dừng.

In [12]:
def is_perfect_square_newton(num: int) -> bool:
    x = num
    while x * x > num:
        x = (x + num // x) // 2
    return x * x == num

def trace_newton(num):
    x = num
    path = [x]
    while x * x > num:
        x = (x + num // x) // 2
        path.append(x)
    print(f"num={num}: dãy x = {path} -> x*x={x*x} -> {x * x == num}")

trace_newton(16)
trace_newton(14)
trace_newton(2147395600)

num=16: dãy x = [16, 8, 5, 4] -> x*x=16 -> True
num=14: dãy x = [14, 7, 4, 3] -> x*x=9 -> False
num=2147395600: dãy x = [2147395600, 1073697800, 536848901, 268424452, 134212229, 67106122, 33553076, 16776569, 8388348, 4194301, 2097406, 1049214, 525630, 264857, 136482, 76107, 52161, 46664, 46341, 46340] -> x*x=2147395600 -> True


Số bước của Newton với `num = 2·10^9` chỉ khoảng 20 (xem dãy ở trên), và gần đáp án thì mỗi bước **gấp đôi số chữ số đúng** (hội tụ bậc hai). Trong phỏng vấn, đây là "điểm cộng"; còn binary search là lời giải chuẩn, dễ chứng minh.

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def isPerfectSquare(self, num: int) -> bool:
        lo, hi = 1, num
        while lo <= hi:
            mid = (lo + hi) // 2
            sq = mid * mid
            if sq == num:
                return True
            if sq < num:
                lo = mid + 1
            else:
                hi = mid - 1
        return False


sol = Solution()
print(sol.isPerfectSquare(16), sol.isPerfectSquare(14))

True False


Bản nộp thay thế (Newton, ngắn):

```python
class Solution:
    def isPerfectSquare(self, num: int) -> bool:
        x = num
        while x * x > num:
            x = (x + num // x) // 2
        return x * x == num
```

## 9. Vì sao đúng?

**Binary search — invariant:** *nếu tồn tại `k` với `k*k == num` thì `lo ≤ k ≤ hi`.*

- Ban đầu `[1, num]` chứa mọi ứng viên (vì `num ≥ 1` nên `1 ≤ k ≤ num`).
- Nếu `mid² < num` thì mọi `k ≤ mid` có `k² ≤ mid² < num` (hàm bình phương tăng trên số dương) → `k` không thể `≤ mid` → đặt `lo = mid + 1` vẫn giữ invariant. Trường hợp `>` đối xứng.
- Mỗi vòng khoảng `[lo, hi]` **thu hẹp ít nhất 1** (do `+1`/`-1`) → chắc chắn dừng.
- Dừng với `lo > hi`: khoảng rỗng mà invariant vẫn đúng → không có `k` → `False` là đúng.

**Tổng số lẻ:** sau `j` lần trừ, `num` đã giảm đúng `j²`. Vòng dừng ở `j` đầu tiên có `j² ≥ num_gốc`; kết quả `0` ⇔ `j² = num_gốc`.

**Newton nguyên:** khi `x > ⌊√num⌋` thì `x_mới < x` và `x_mới ≥ ⌊√num⌋` (bất đẳng thức AM-GM), nên dãy giảm ngặt và dừng đúng tại `⌊√num⌋`.

## 10. Độ phức tạp

Gọi `n = num`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử lần lượt | `O(√n)` | `O(1)` | ~46 341 vòng với `n` lớn nhất |
| 2. Binary search | `O(log n)` | `O(1)` | ~31 vòng; **lời giải chuẩn** |
| 3. Tổng số lẻ | `O(√n)` | `O(1)` | chỉ dùng phép trừ |
| 4. Newton | `O(log n)` (thực tế rất ít vòng) | `O(1)` | hội tụ bậc hai |
| (cấm) `num ** 0.5` | `O(1)` | `O(1)` | rủi ro sai số float, vi phạm đề |

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
methods = {
    "linear": is_perfect_square_linear,
    "binary": is_perfect_square_binary,
    "odd": is_perfect_square_odd,
    "newton": is_perfect_square_newton,
    "Solution": Solution().isPerfectSquare,
}
cases = [
    (16, True), (14, False),                      # mẫu đề
    (1, True), (2, False), (3, False), (4, True), # biên nhỏ
    (808201, True), (808200, False),             # 899^2 và 899^2 - 1
    (2147395600, True),                           # 46340^2 — số chính phương lớn nhất <= 2^31-1
    (2147483647, False),                          # 2^31 - 1
    (2147395601, False),                          # ngay sau một số chính phương
]
for num, exp in cases:
    for name, f in methods.items():
        assert f(num) == exp, (name, num)
    print(f"num={num:>10} -> {exp} PASS")

num=        16 -> True PASS
num=        14 -> False PASS
num=         1 -> True PASS
num=         2 -> False PASS
num=         3 -> False PASS
num=         4 -> True PASS
num=    808201 -> True PASS
num=    808200 -> False PASS
num=2147395600 -> True PASS
num=2147483647 -> False PASS
num=2147395601 -> False PASS


In [15]:
# Kiểm thử toàn bộ miền nhỏ 1..5000 và ngẫu nhiên có seed trên miền lớn (oracle: math.isqrt chỉ dùng trong test)
for n in range(1, 5001):
    exp = math.isqrt(n) ** 2 == n
    for name in ["linear", "binary", "odd", "newton"]:
        assert methods[name](n) == exp, (name, n)
import random
rng = random.Random(367)
for _ in range(2000):
    n = rng.randint(1, 2**31 - 1)
    if rng.random() < 0.5:
        n = rng.randint(1, 46340) ** 2          # một nửa test là số chính phương
    exp = math.isqrt(n) ** 2 == n
    for name in ["binary", "newton", "Solution"]:
        assert methods[name](n) == exp, (name, n)
print("Miền 1..5000 và 2000 test ngẫu nhiên PASS")

Miền 1..5000 và 2000 test ngẫu nhiên PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Cập nhật `lo = mid` (thiếu `+1`) → lặp vô hạn
Khi `hi = lo + 1`, `mid = lo`; nếu `mid² < num` thì `lo = mid` không đổi → kẹt mãi.

### Lỗi 2 — Điều kiện `while lo < hi` nhưng vẫn cập nhật `hi = mid - 1`
Bỏ sót trường hợp khoảng còn **đúng 1 phần tử** chưa kiểm tra. Ví dụ `num = 4`: `[1,4]` → `mid=2` → `4 == 4` OK; nhưng `num = 1` thì `lo = hi = 1` → vòng lặp không chạy → trả `False` sai.

### Lỗi 3 — Dùng float
`int(num ** 0.5) ** 2 == num` thường đúng với constraints này nhưng vi phạm yêu cầu "không dùng sqrt"; với số lớn hơn có thể sai (xem mục 2.2).

### Lỗi 4 — Tràn số ở ngôn ngữ khác
Trong C++/Java, `mid * mid` có thể tràn `int` và `(lo + hi)` cũng có thể tràn. Python không tràn, nhưng nên biết cách viết `mid = lo + (hi - lo) // 2` và dùng kiểu 64-bit.

In [16]:
def wrong_lt(num):
    lo, hi = 1, num
    while lo < hi:                  # thiếu trường hợp lo == hi
        mid = (lo + hi) // 2
        if mid * mid == num:
            return True
        if mid * mid < num:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

def wrong_infinite(num, max_steps=50):
    lo, hi = 1, num
    steps = 0
    while lo <= hi:
        steps += 1
        if steps > max_steps:
            return "LẶP VÔ HẠN (đã chặn sau 50 bước)"
        mid = (lo + hi) // 2
        if mid * mid == num:
            return True
        if mid * mid < num:
            lo = mid              # thiếu +1
        else:
            hi = mid - 1
    return False

print("Lỗi 2 — wrong_lt(1) =", wrong_lt(1), "(đúng: True)")
print("Lỗi 1 — wrong_infinite(5) =", wrong_infinite(5))
print("lo + (hi - lo) // 2 với lo=3, hi=10:", 3 + (10 - 3) // 2, "== (3+10)//2 =", (3 + 10) // 2)

Lỗi 2 — wrong_lt(1) = False (đúng: True)
Lỗi 1 — wrong_infinite(5) = LẶP VÔ HẠN (đã chặn sau 50 bước)
lo + (hi - lo) // 2 với lo=3, hi=10: 6 == (3+10)//2 = 6


## 13. Mẹo nhận dạng pattern

- Đề hỏi "có tồn tại số nguyên `k` sao cho `f(k) = target`" với **`f` tăng dần** → **binary search trên đáp án**.
- Từ khóa "không dùng sqrt/pow", "căn bậc hai nguyên" → binary search hoặc Newton.
- Template binary search "tìm chính xác": `while lo <= hi`, `lo = mid + 1`, `hi = mid - 1`.
- Template "tìm biên" (giá trị lớn nhất có `f(k) ≤ target`): sau vòng lặp, `hi` là đáp án (xem bài 69 Sqrt(x)).
- Bài anh em: **69 Sqrt(x)**, **374 Guess Number Higher or Lower**, **278 First Bad Version**, **633 Sum of Square Numbers**.

## 14. Biến thể

### 14.1. Sqrt(x) — trả `⌊√x⌋` (LeetCode 69)

Dùng chính vòng binary search; khi kết thúc `lo > hi`, `hi` là số lớn nhất có `hi² ≤ x`.

In [17]:
def my_sqrt(x: int) -> int:
    lo, hi = 0, x
    while lo <= hi:
        mid = (lo + hi) // 2
        if mid * mid <= x:
            lo = mid + 1
        else:
            hi = mid - 1
    return hi

print([my_sqrt(x) for x in [0, 1, 4, 8, 15, 16, 2147395600]])

[0, 1, 2, 2, 3, 4, 46340]


### 14.2. Lập phương hoàn hảo (perfect cube)

Thay `mid * mid` bằng `mid ** 3` — mọi thứ khác giữ nguyên. Đây là sức mạnh của template "tìm trên đáp án".

In [18]:
def is_perfect_cube(num: int) -> bool:
    lo, hi = 1, num
    while lo <= hi:
        mid = (lo + hi) // 2
        c = mid ** 3
        if c == num:
            return True
        if c < num:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

print([n for n in range(1, 1001) if is_perfect_cube(n)])

[1, 8, 27, 64, 125, 216, 343, 512, 729, 1000]


## 15. Bài tập tự luyện (TODO)

**Bài 1 (tính tay).** Lập bảng trace binary search cho `num = 49` và `num = 50`.

**Bài 2 (dự đoán output).** Dãy `x` của Newton với `num = 100` là gì?

**Bài 3 (điền chỗ trống).** Hoàn thiện `exercise_binary`.

**Bài 4 (viết lại).** Viết `exercise_odd` theo ý tưởng tổng số lẻ, nhưng **không sửa** biến `num` (dùng biến `total` cộng dần).

**Bài 5 (biến thể).** Viết `count_squares_upto(n)`: đếm số chính phương trong `[1, n]` **không** dùng `sqrt`.

**Bài 6 (mini challenge — LeetCode 633).** `judge_square_sum(c)`: có tồn tại `a, b ≥ 0` nguyên với `a² + b² = c` không? Gợi ý: hai con trỏ hoặc dùng `is_perfect_square`.

In [19]:
def exercise_binary(num: int) -> bool:
    # TODO 1: lo, hi = 1, num
    # TODO 2: while lo <= hi: tính mid, sq
    # TODO 3: ba nhánh ==, <, >
    pass

# assert exercise_binary(49) and not exercise_binary(50)

In [20]:
def exercise_odd(num: int) -> bool:
    # TODO: total = 0, odd = 1; cộng odd vào total cho tới khi total >= num
    pass

# assert exercise_odd(49) and not exercise_odd(50)

In [21]:
def count_squares_upto(n: int) -> int:
    # TODO
    pass

# assert count_squares_upto(1) == 1 and count_squares_upto(15) == 3 and count_squares_upto(16) == 4

In [22]:
def judge_square_sum(c: int) -> bool:
    # TODO
    pass

# assert judge_square_sum(5) and not judge_square_sum(3) and judge_square_sum(0) and judge_square_sum(2)

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | bắt đầu `[1, num]` | `mid = (lo + hi) // 2` | 49: mid=25 → 12 → 6 → 9 → 7 ✔ |
| 2 | `x = (x + num // x) // 2` | bắt đầu `x = 100` | 100 → 50 → 26 → ... |
| 3 | xem mục 4 | `lo = mid + 1` / `hi = mid - 1` | `return False` sau vòng lặp |
| 4 | `total` là tổng các số lẻ đã cộng | `while total < num` | `return total == num` |
| 5 | số chính phương ≤ n là `1², ..., k²` | đáp án là `⌊√n⌋` | dùng `my_sqrt(n)` (binary search) |
| 6 | `a` từ 0, `b` từ `⌊√c⌋` | tổng nhỏ → tăng `a`; lớn → giảm `b` | `while a <= b` |

## 17. Đáp án tham khảo

In [23]:
def answer_binary(num):
    lo, hi = 1, num
    while lo <= hi:
        mid = (lo + hi) // 2
        sq = mid * mid
        if sq == num:
            return True
        if sq < num:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

def answer_odd(num):
    total, odd = 0, 1
    while total < num:
        total += odd
        odd += 2
    return total == num

def answer_count_squares_upto(n):
    return my_sqrt(n)

def answer_judge_square_sum(c):
    a, b = 0, my_sqrt(c)
    while a <= b:
        s = a * a + b * b
        if s == c:
            return True
        if s < c:
            a += 1
        else:
            b -= 1
    return False

print("Bài 1:")
_ = trace_binary(49); print(); _ = trace_binary(50)
print("Bài 2:"); trace_newton(100)
assert answer_binary(49) and not answer_binary(50)
assert answer_odd(49) and not answer_odd(50)
assert answer_count_squares_upto(1) == 1 and answer_count_squares_upto(15) == 3 and answer_count_squares_upto(16) == 4
assert answer_judge_square_sum(5) and not answer_judge_square_sum(3) and answer_judge_square_sum(0) and answer_judge_square_sum(2)
print("Đáp án PASS toàn bộ!")

Bài 1:
num = 49
Bước 1: lo=1, hi=49, mid=25, mid²=625 > num -> hi = 24
Bước 2: lo=1, hi=24, mid=12, mid²=144 > num -> hi = 11
Bước 3: lo=1, hi=11, mid=6, mid²=36 < num -> lo = 7
Bước 4: lo=7, hi=11, mid=9, mid²=81 > num -> hi = 8
Bước 5: lo=7, hi=8, mid=7, mid²=49 == num -> True

num = 50
Bước 1: lo=1, hi=50, mid=25, mid²=625 > num -> hi = 24
Bước 2: lo=1, hi=24, mid=12, mid²=144 > num -> hi = 11
Bước 3: lo=1, hi=11, mid=6, mid²=36 < num -> lo = 7
Bước 4: lo=7, hi=11, mid=9, mid²=81 > num -> hi = 8
Bước 5: lo=7, hi=8, mid=7, mid²=49 < num -> lo = 8
Bước 6: lo=8, hi=8, mid=8, mid²=64 > num -> hi = 7
Dừng: lo=8 > hi=7 -> False
Bài 2:
num=100: dãy x = [100, 50, 26, 14, 10] -> x*x=100 -> True
Đáp án PASS toàn bộ!


## 18. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def isPerfectSquare(self, num: int) -> bool:
        lo, hi = 1, num
        while lo <= hi:                 # khoảng [lo, hi] còn ứng viên
            mid = (lo + hi) // 2
            sq = mid * mid
            if sq == num:
                return True
            if sq < num:
                lo = mid + 1            # bỏ nửa trái kể cả mid
            else:
                hi = mid - 1            # bỏ nửa phải kể cả mid
        return False
```

**Checklist**
- [ ] Không dùng `sqrt`, `** 0.5`, `math.isqrt` trong code nộp?
- [ ] `while lo <= hi` (có dấu `=`) để không bỏ sót khoảng 1 phần tử (`num = 1`)?
- [ ] Cập nhật `lo = mid + 1` / `hi = mid - 1` để tránh lặp vô hạn?
- [ ] Dùng `//` (chia nguyên), không dùng `/`?
- [ ] Đã test `1, 2, 3, 4`, số chính phương lớn nhất `2147395600` và `2^31 - 1`?